In [ ]:

import os, socket, time, subprocess, torch

t0 = time.time()
print("PROBE START  ", time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()), "UTC", flush=True)
print("hostname     :", socket.gethostname(), flush=True)
print("torch        :", torch.__version__, flush=True)
print("cuda avail   :", torch.cuda.is_available(), flush=True)
print("device count :", torch.cuda.device_count(), flush=True)
for i in range(torch.cuda.device_count()):
    print(f"  gpu{i}     :", torch.cuda.get_device_name(i),
          f"| {torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB", flush=True)
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu",
                          "--format=csv,noheader"], capture_output=True, text=True,
                         timeout=60).stdout.strip(), flush=True)
except Exception as e:
    print("nvidia-smi failed:", e, flush=True)

# Do a tiny matmul so we know the GPU is actually usable, not just attached.
if torch.cuda.is_available():
    a = torch.randn(4096, 4096, device="cuda")
    for _ in range(20):
        a = a @ a / a.shape[-1]
    torch.cuda.synchronize()
    print("matmul ok, elapsed %.1fs" % (time.time() - t0), flush=True)

SLEEP = 180
print(f"sleeping {SLEEP}s to hold the session open...", flush=True)
time.sleep(SLEEP)
print("PROBE END    ", time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()), "UTC", flush=True)
print("total elapsed %.1fs" % (time.time() - t0), flush=True)
